In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
import torch.optim as optim
from typing import Tuple, Dict

# Best Explanation

https://claude.ai/chat/a265a15a-8d92-40e5-9e72-4a33b8913085

In [12]:
class Expert(nn.Module):
    """SwiGLU FFN expert.

    Used for both routed experts and the shared expert (with different inter_dim).
    SwiGLU: output = W_down(SiLU(W_gate(x)) * W_up(x))
    """

    def __init__(self, hidden_dim: int, inter_dim: int):
        super().__init__()
        self.w_gate = nn.Linear(hidden_dim, inter_dim, bias=False)
        self.w_up = nn.Linear(hidden_dim, inter_dim, bias=False)
        self.w_down = nn.Linear(inter_dim, hidden_dim, bias=False)
    def forward(self, x: Tensor) -> Tensor:
        # x: [N_tokens, D] → [N_tokens, D]
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

In [13]:
# Linear classifier: scores = x @ W_router x: [N, D] W_router: [D, N_experts]

# Expert 5:  receives 40% of all tokens  ← overloaded, becomes "generic"
# Expert 12: receives 35% of all tokens
# Expert 0:  receives 15%
# ...
# Expert 61: receives 0.001%              ← starved, weights go stale
# Expert 62: receives 0.0005%
# Expert 63: receives 0.0002%             ← dead expert

"""
DeepSeek V3's solution: Group-based routing.
# """
# Organize 64 experts into 8 groups of 8:

# Group 0: [Expert 0,  1,  2,  3,  4,  5,  6,  7 ]
# Group 1: [Expert 8,  9,  10, 11, 12, 13, 14, 15]
# Group 2: [Expert 16, 17, 18, 19, 20, 21, 22, 23]
# ...
# Group 7: [Expert 56, 57, 58, 59, 60, 61, 62, 63]

"""
2 Stage Routing
"""
# Stage 1: Pick top-4 groups (of 8) — "which departments are relevant?"
# Within those 4 groups (32 candidate experts), pick top-8 — "which specialists?"


'\n2 Stage Routing\n'

## The Full Gate Pipeline — One Diagram
```
x [N, D]
    │
    ▼
W_router [D, E] ──→ logits [N, E] ──→ sigmoid ──→ scores [N, E]  (FP32)
                                                      │
                                    ┌─────────────────┼──────────────────┐
                                    ▼                 ▼                  ▼
                              SELECTION PATH    WEIGHTING PATH     MONITORING
                                    │                 │                  │
                              view as [N,G,EPG]       │            one_hot [N,E]
                              top2-sum per group      │            load_counts [E]
                              → group_scores [N,G]    │            aux_loss, H_load
                              topk groups [N,topk_g]  │
                              → expert_mask [N,E]     │
                              scores + bias           │
                              mask non-selected       │
                              topk → indices [N,K]    │
                                    │                 │
                                    └────► gather ◄───┘
                                           │
                                     weights [N,K] (original scores at selected indices)
                                           │
                                     normalize (sum=1)
                                           │
                                     × route_scale
                                           │
                                     weights [N,K]  ← final

In [16]:
class Gate(nn.Module):
    """MoE Router with DeepSeek V3 auxiliary-loss-free load balancing.

    Key design (from DeepSeek V3 official code):
    1. Sigmoid scoring — independent expert probabilities (not competitive softmax)
    2. Group-based routing — top-2-sum per group, select topk_group groups
    3. Bias for selection, original scores for weights — bias steers without corrupting gradients
    4. FP32 for all gating computations — numerical stability
    5. norm_topk_prob — normalize selected weights to sum to 1 (BUG FIX #2)
    6. routed_scaling_factor applied to weights after normalization
    """
    def __init__(
        self,
        hidden_dim: int,
        n_routed_experts: int,
        num_experts_per_tok: int,
        n_group: int,
        topk_group: int,
        scoring_func: str = "sigmoid",
        norm_topk_prob: bool = True,
        routed_scaling_factor: float = 2.5,
        seq_aux_loss_alpha: float = 0.0001,
    ):
        super().__init__()
        self.n_routed_experts = n_routed_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.n_group = n_group
        self.topk_group = topk_group
        self.experts_per_group = n_routed_experts // n_group
        self.scoring_func = scoring_func
        self.norm_topk_prob = norm_topk_prob
        self.routed_scaling_factor = routed_scaling_factor
        self.seq_aux_loss_alpha = seq_aux_loss_alpha

        # Router projection: hidden_dim → n_routed_experts
        self.router_weight = nn.Linear(hidden_dim, n_routed_experts, bias=False)

        # Dynamic bias for auxiliary-loss-free load balancing (NOT a parameter)
        self.register_buffer("bias", torch.zeros(n_routed_experts))

    def forward(self, hidden_states: Tensor) -> Tuple[Tensor, Tensor, Tensor, Dict[str, Tensor]]:
        """Route tokens to experts.

        Args:
            hidden_states: [N, D] (pre-flattened by MoE, N = B*S)

        Returns:
            weights:  [N, K] — routing weights (scaled, normalized original scores)
            indices:  [N, K] — selected expert indices
            aux_loss: scalar — sequence-level auxiliary loss
            metadata: dict with 'load_counts' [E] and 'H_load' scalar
        """
        N = hidden_states.shape[0] # number of tokens
        E = self.n_routed_experts # number of experts
        K = self.num_experts_per_tok # number of experts per token
        G = self.n_group # number of groups
        EPG = self.experts_per_group # number of experts per group
        dtype = hidden_states.dtype
        device = hidden_states.device

        logits = self.router_weight(hidden_states) # [N, D] @ [D, E] → [N, E]
        if self.scoring_func == "sigmoid":
            scores = torch.sigmoid(logits) # [N, E]
        elif self.scoring_func == "softmax":
            scores = F.softmax(logits, dim=-1) # [N, E]
        else:
            raise ValueError(f"Invalid scoring function: {self.scoring_func}")
        
        # x: [4, D] → logits: [4, 16] → sigmoid → scores: [4, 16]

        # scores (FP32):
        # each token has 16 experts organized into 4 groups of 4 experts
        
        # Num_tokens, num_groups, num_experts_per_group
        # Token 0: [0.82, 0.15, 0.67, 0.91, │ 0.23, 0.88, 0.12, 0.45, │ 0.71, 0.33, 0.56, 0.19, │ 0.44, 0.68, 0.77, 0.29]
        # Token 1: [0.11, 0.73, 0.55, 0.22, │ 0.89, 0.14, 0.76, 0.63, │ 0.31, 0.82, 0.47, 0.15, │ 0.58, 0.21, 0.69, 0.43]
        # Token 2: [0.45, 0.38, 0.81, 0.29, │ 0.17, 0.52, 0.93, 0.44, │ 0.66, 0.25, 0.71, 0.88, │ 0.13, 0.57, 0.34, 0.72]
        # Token 3: [0.33, 0.91, 0.18, 0.74, │ 0.55, 0.27, 0.83, 0.41, │ 0.19, 0.67, 0.45, 0.78, │ 0.86, 0.22, 0.61, 0.39]
        #           ├── Group 0 ──────────┤  ├── Group 1 ──────────┤  ├── Group 2 ──────────┤  ├── Group 3 ──────────┤
        
        # Step 2: Group based routing
        # score each group by sum of its top-2 expert scores
        scores_grouped = scores.view(N, G, EPG) # [N, G, EPG]
        group_top2, _ = scores_grouped.topk(2, dim=-1) # [N, G, 2]
        group_scores = group_top2.sum(dim=-1) # [N, G]
        
        # For Token 0:
        # ```
        # scores_grouped[0]:
        #   Group 0: [0.82, 0.15, 0.67, 0.91]  → top2 = [0.91, 0.82] → sum = 1.73 ✓
        #   Group 1: [0.23, 0.88, 0.12, 0.45]  → top2 = [0.88, 0.45] → sum = 1.33
        #   Group 2: [0.71, 0.33, 0.56, 0.19]  → top2 = [0.71, 0.56] → sum = 1.27
        #   Group 3: [0.44, 0.68, 0.77, 0.29]  → top2 = [0.77, 0.68] → sum = 1.45

        # group_scores[0] = [1.73, 1.33, 1.27, 1.45] across 4 groups

        # now we pick top 2 groups with highest scores 1.73 and 1.45
        #```

        _, top_groups = group_scores.topk(topk_group, dim=-1) # [4, 2]
        # ```
        # Token 0: group_scores = [1.73, 1.33, 1.27, 1.45]
        #          top_groups = [0, 3]  ← Group 0 (1.73) and Group 3 (1.45)

        # This means Token 0 can ONLY choose experts from Groups 0 and 3.
        # Experts 4-11 (Groups 1,2) are MASKED OUT entirely.

        group_mask = torch.zeros(N, G, device=device, dtype=dtype)
        group_mask.scatter_(1, top_groups, 1) # [N, G]
        expert_mask = group_mask.unsqueeze(-1).expand(-1,-1,EPG).reshape(N, E)  # [4, 16]

        # Token 0:
        #   group_mask = [1, 0, 0, 1]   ← Groups 0 and 3 selected
        
        #   expert_mask = [1,1,1,1, 0,0,0,0, 0,0,0,0, 1,1,1,1]
        #                  Group 0   Group 1   Group 2   Group 3
        #                  ✓ open     ✗ masked  ✗ masked  ✓ open

        # STEP 3: Biased selection (bias NOT in gradient graph)
        biased = scores + self.bias                              # [N, E]
        biased = biased.masked_fill(expert_mask == 0, -1e9)      # kill masked groups
        _, indices = biased.topk(K, dim=-1)                      # [N, K]
        # ```
        # Token 0:
        #   scores:  [0.82, 0.15, 0.67, 0.91, │ ..., │ ..., │ 0.44, 0.68, 0.77, 0.29]
        #   bias:    [0.01,-0.05, 0.02, 0.00,  │ ..., │ ..., │ 0.03, 0.01,-0.02, 0.04]
        #   biased:  [0.83, 0.10, 0.69, 0.91, │-1e9, │-1e9, │ 0.47, 0.69, 0.75, 0.33]
        #                                        masked out!

        #   topk(4): indices = [3, 0, 14, 2]  ← Expert 3(0.91), Expert 0(0.83), Expert 14(0.75), Expert 2(0.69)
        # ```
        weights = scores.gather(1, indices)                       # [N, K]
        # Token 0:
        #   indices = [3, 0, 14, 2]
        #   weights = [scores[3], scores[0], scores[14], scores[2]]
        #           = [0.91,      0.82,      0.77,       0.67]
        #                                     ↑
        #                           Original score! NOT 0.75 (biased)
        if norm_topk_prob:
            weights = weights / (weights.sum(dim=-1, keepdim=True) + 1e-20)
        # Token 0:
        #   weights = [0.91, 0.82, 0.77, 0.67]
        #   sum = 3.17
        #   normalized = [0.287, 0.259, 0.243, 0.211]
        #   sum check: 0.287 + 0.259 + 0.243 + 0.211 = 1.000 ✓

        # STEP: Apply scaling factor
        weights = weights * self.routed_scaling_factor
        # Token 0:
        # weights = [0.287, 0.259, 0.243, 0.211] × 2.5
        #         = [0.718, 0.647, 0.608, 0.528]
        # ``

        # STEP 8: Compute auxiliary loss and metadata
        one_hot = torch.zeros(N, E, device=device, dtype=scores.dtype)
        one_hot.scatter_(1, indices, 1.0)
        load_counts = one_hot.sum(dim=0)  # [E]
        #  Sequence-level aux loss: L = alpha * mean((f_i - 1/E)^2)
        f = load_counts / max(N, 1)
        target = 1.0 / E
        aux_loss = self.seq_aux_loss_alpha * ((f - target) ** 2).mean()

        # Load balance entropy for monitoring (H_load → higher = more balanced)
        f_safe = f + 1e-10
        H_load = -(f_safe * f_safe.log()).sum()

        metadata = {
            "load_counts": load_counts.detach(),
            "H_load": H_load.detach(),
        }

        # Cast weights back to input dtype
        weights = weights.to(dtype)

        return weights, indices, aux_loss, metadata

    def update_bias(self, load_counts: Tensor, gamma: float) -> None:
        """Update dynamic bias for load balancing (called by training loop).

        Args:
            load_counts: [E] — token counts per expert from forward pass
            gamma: bias update rate (0.0 if frozen)
        """
        if gamma <= 0:
            return
        mean_load = load_counts.float().mean()
        if mean_load > 0:
            self.bias -= gamma * (load_counts.float() - mean_load) / mean_load



In [18]:
class MoE(nn.Module):
    """Mixture of Experts with DeepSeek V3 architecture.

    Components:
    - Gate (router): sigmoid scoring + group routing + bias balancing
    - Routed experts: 64 SwiGLU FFNs, top-8 selected per token
    - Shared expert: 1 SwiGLU MLP with combined inter_dim (n_shared * moe_inter_dim)
      Applied to ALL tokens unconditionally (captures common patterns)

    Token-centric dispatch: iterate over experts, process each expert's batch.
    """

    def __init__(
        self,
        hidden_dim: int,
        moe_inter_dim: int,
        n_routed_experts: int = 64,
        num_experts_per_tok: int = 8,
        n_shared_experts: int = 2,
        n_group: int = 8,
        topk_group: int = 4,
        scoring_func: str = "sigmoid",
        norm_topk_prob: bool = True,
        routed_scaling_factor: float = 2.5,
        seq_aux_loss_alpha: float = 0.0001,
        shared_inter_dim: Optional[int] = None,
    ):
        super().__init__()
        self.n_routed_experts = n_routed_experts
        self.num_experts_per_tok = num_experts_per_tok

        # Gate (router)
        self.gate = Gate(
            hidden_dim=hidden_dim,
            n_routed_experts=n_routed_experts,
            num_experts_per_tok=num_experts_per_tok,
            n_group=n_group,
            topk_group=topk_group,
            scoring_func=scoring_func,
            norm_topk_prob=norm_topk_prob,
            routed_scaling_factor=routed_scaling_factor,
            seq_aux_loss_alpha=seq_aux_loss_alpha,
        )

        # Routed experts (sparse, selected per token)
        self.routed_experts = nn.ModuleList([
            Expert(hidden_dim, moe_inter_dim) for _ in range(n_routed_experts)
        ])

        # Shared expert (dense, always active) — single MLP with combined inter_dim
        # DeepSeek V3 pattern: n_shared_experts is a multiplier, not a count of modules
        effective_shared_dim = shared_inter_dim or (n_shared_experts * moe_inter_dim)
        self.shared_expert = Expert(hidden_dim, effective_shared_dim)

    def forward(self, hidden_states: Tensor) -> Tuple[Tensor, Dict[str, Tensor]]:
            """Process tokens through routed + shared experts.

            Args:
                hidden_states: [B, S, D]

            Returns:
                output:   [B, S, D]
                aux_data: dict with 'aux_loss', 'load_counts', 'H_load'
            """
            orig_shape = hidden_states.shape  # [B, S, D]
            hidden_dim = orig_shape[-1]
            x = hidden_states.view(-1, hidden_dim)  # [N, D] where N = B*S
            # Route tokens
            weights, indices, aux_loss, metadata = self.gate(x)
            # weights: [N, K], indices: [N, K]

            # Token-centric dispatch for routed experts
            routed_output = torch.zeros_like(x) # [N, K]
            for expert_idx in range(self.n_routed_experts):
                # find the indices of the tokens that are routed to this expert
                # Which positions in indices equal 0?
                mask = (indices == 0)
                # indices: [[0,2], [1,3], [0,1], [2,3], [0,3], [1,2]]
                # mask:    [[T,F], [F,F], [T,F], [F,F], [T,F], [F,F]]     bool [N, K]
                # Which TOKENS have at least one True?
                token_mask = mask.any(dim=-1)
                # [True, False, True, False, True, False]     bool [N]
                
                if not token_mask.any():
                    continue  # Skip experts with 0 assigned tokens
                
                # Process this expert's batch
                expert_input = x[token_mask]  # [n_tokens, D]
                # = x[[0, 2, 4]] = [[x₀], [x₂], [x₄]]     shape: [3, D]
                expert_out = self.routed_experts[expert_idx](expert_input)  # [n_tokens, D]
                # = [[e₀(x₀)], [e₀(x₂)], [e₀(x₄)]]       shape: [3, D]

                # Sum weights across all slots that selected this expert per token
                # (handles rare case where same expert selected in multiple slots)
                # weights: [[0.6,0.4], [0.7,0.3], [0.5,0.5], [0.4,0.6], [0.8,0.2], [0.3,0.7]]
                # mask:    [[T,  F  ], [F,  F  ], [T,  F  ], [F,  F  ], [T,  F  ], [F,  F  ]]

                # Element-wise: weights * mask.float()
                # =            [[0.6, 0 ], [0,   0 ], [0.5, 0 ], [0,   0 ], [0.8, 0 ], [0,   0 ]]

                # Sum across K dimension:
                # (weights * mask.float()).sum(dim=-1)
                # = [0.6, 0.0, 0.5, 0.0, 0.8, 0.0]     shape: [N]

                # Select only the tokens that went to this expert:
                expert_weights = (weights * mask.float()).sum(dim=-1)[token_mask]
                # = [0.6, 0.5, 0.8]     shape: [3]   ← one weight per token
                
                active_weights = expert_weights[token_mask]  # [n_tokens]

                routed_output[token_mask] += expert_out * active_weights.unsqueeze(-1)

            # Shared expert — processes ALL tokens
            shared_output = self.shared_expert(x)  # [N, D]

            # Combine routed + shared
            output = routed_output + shared_output  # [N, D]
            output = output.view(*orig_shape)  # [B, S, D]

            aux_data = {
                "aux_loss": aux_loss,
                "load_counts": metadata["load_counts"],
                "H_load": metadata["H_load"],
            }

            return output, aux_data


NameError: name 'Optional' is not defined

In [19]:
# expert_out: [3, D]   (expert 0's output for tokens 0, 2, 4)
# expert_w:   [3]      (weights: 0.6, 0.5, 0.8)

# Weight each token's output:
# expert_w.unsqueeze(-1): [3, 1]   (broadcast across D dimension)
# expert_out * expert_w.unsqueeze(-1): [3, D]
"""
routed_output[token_mask] += expert_out * expert_w.unsqueeze(-1)
```
```
routed_output (initially all zeros [6, D]):

After Expert 0:
  routed_output[0] += 0.6 × e₀(x₀)
  routed_output[2] += 0.5 × e₀(x₂)
  routed_output[4] += 0.8 × e₀(x₄)

After Expert 1 (tokens 1, 2, 5):
  routed_output[1] += 0.7 × e₁(x₁)
  routed_output[2] += 0.5 × e₁(x₂)    ← Token 2 ACCUMULATES from both experts!
  routed_output[5] += 0.3 × e₁(x₅)

After Expert 2 (tokens 0, 3, 5):
  routed_output[0] += 0.4 × e₂(x₀)    ← Token 0 now has contributions from Expert 0 AND Expert 2
  routed_output[3] += 0.4 × e₂(x₃)
  routed_output[5] += 0.7 × e₂(x₅)

After Expert 3 (tokens 1, 3, 4):
  routed_output[1] += 0.3 × e₃(x₁)
  routed_output[3] += 0.6 × e₃(x₃)
  routed_output[4] += 0.2 × e₃(x₄)
```

Final state for Token 0:
```
routed_output[0] = 0.6 × e₀(x₀) + 0.4 × e₂(x₀)
                   ├── Expert 0 ──┤  ├── Expert 2 ──┤
                   
This is the weighted mixture of experts — the CORE idea of MoE.
```

*Kết quả cuối cho Token 0: tổng trọng số của đầu ra từ Expert 0 (trọng số 0.6) và Expert 2 (trọng số 0.4). Đây chính xác là "hỗn hợp chuyên gia" — ý tưởng cốt lõi của MoE.*

### Visual summary of the full dispatch
```
Token:     0         1         2         3         4         5
Experts: [0,2]     [1,3]     [0,1]     [2,3]     [0,3]     [1,2]
Weights: [.6,.4]   [.7,.3]   [.5,.5]   [.4,.6]   [.8,.2]   [.3,.7]

Expert 0 processes: ──→ {Token 0, Token 2, Token 4}  batch=[3, D]
Expert 1 processes: ──→ {Token 1, Token 2, Token 5}  batch=[3, D]
Expert 2 processes: ──→ {Token 0, Token 3, Token 5}  batch=[3, D]
Expert 3 processes: ──→ {Token 1, Token 3, Token 4}  batch=[3, D]

                    ┌── 0.6·e₀(x₀) + 0.4·e₂(x₀) ──→ Token 0 output
                    ├── 0.7·e₁(x₁) + 0.3·e₃(x₁) ──→ Token 1 output
 Accumulate into    ├── 0.5·e₀(x₂) + 0.5·e₁(x₂) ──→ Token 2 output
 routed_output:     ├── 0.4·e₂(x₃) + 0.6·e₃(x₃) ──→ Token 3 output
                    ├── 0.8·e₀(x₄) + 0.2·e₃(x₄) ──→ Token 4 output
                    └── 0.3·e₁(x₅) + 0.7·e₂(x₅) ──→ Token 5 output
"""

'\nrouted_output[token_mask] += expert_out * expert_w.unsqueeze(-1)\n```\n```\nrouted_output (initially all zeros [6, D]):\n\nAfter Expert 0:\n  routed_output[0] += 0.6 × e₀(x₀)\n  routed_output[2] += 0.5 × e₀(x₂)\n  routed_output[4] += 0.8 × e₀(x₄)\n\nAfter Expert 1 (tokens 1, 2, 5):\n  routed_output[1] += 0.7 × e₁(x₁)\n  routed_output[2] += 0.5 × e₁(x₂)    ← Token 2 ACCUMULATES from both experts!\n  routed_output[5] += 0.3 × e₁(x₅)\n\nAfter Expert 2 (tokens 0, 3, 5):\n  routed_output[0] += 0.4 × e₂(x₀)    ← Token 0 now has contributions from Expert 0 AND Expert 2\n  routed_output[3] += 0.4 × e₂(x₃)\n  routed_output[5] += 0.7 × e₂(x₅)\n\nAfter Expert 3 (tokens 1, 3, 4):\n  routed_output[1] += 0.3 × e₃(x₁)\n  routed_output[3] += 0.6 × e₃(x₃)\n  routed_output[4] += 0.2 × e₃(x₄)\n```\n\nFinal state for Token 0:\n```\nrouted_output[0] = 0.6 × e₀(x₀) + 0.4 × e₂(x₀)\n                   ├── Expert 0 ──┤  ├── Expert 2 ──┤\n\nThis is the weighted mixture of experts — the CORE idea of MoE.\n```

In [ ]:
"""
shared_output = self.shared_expert(x)     # [N, D] — ALL tokens, one big batch
output = routed_output + shared_output    # [N, D]
```
```
Final output for Token 0:
  = routed_output[0]           +  shared_output[0]
  = 0.6·e₀(x₀) + 0.4·e₂(x₀) +  e_shared(x₀)
    ├── sparse (2 of 64) ─────┤   ├── dense (all tokens) ──┤
```

**Why a shared expert?** Some knowledge is **universal** — it applies to every token regardless of routing. The shared expert captures this common knowledge, freeing routed experts to specialize.

*Tại sao shared expert? Một số kiến thức là PHỔ QUÁT — áp dụng cho mọi token bất kể routing. Shared expert nắm bắt kiến thức chung, giải phóng routed expert để chuyên môn hóa.*

### Shared expert sizing (BUG FIX #5)

From the plan: `n_shared_experts=2` means ONE Expert instance with `inter_dim = 2 × moe_inter_dim`.
```
NOT this (wrong — naive interpretation):
  shared_expert_0 = Expert(2048, 768)   ← two separate MLPs
  shared_expert_1 = Expert(2048, 768)
  shared_out = (shared_expert_0(x) + shared_expert_1(x)) / 2

THIS (correct — DeepSeek V3 actual implementation):
  shared_expert = Expert(2048, 768 * 2)   ← ONE MLP with double inter_dim
  shared_out = shared_expert(x)

Why? One big MLP with inter_dim=1536 is:
  - More parameter-efficient (shared W_down)
  - Better GPU utilization (one large matmul vs two small ones)
  - How DeepSeek V3 actually does it
"""

In [17]:
# =============================================================================
# SECTION 5: MIXTURE OF EXPERTS (MOE)
# =============================================================================
# DeepSeek V3 MoE: sigmoid scoring, group-based routing, auxiliary-loss-free
# bias balancing, shared expert (single MLP with combined inter_dim).
# Reference: DeepSeek-V3 Technical Report Section 3.2


class Expert(nn.Module):
    """SwiGLU FFN expert.

    Used for both routed experts and the shared expert (with different inter_dim).
    SwiGLU: output = W_down(SiLU(W_gate(x)) * W_up(x))
    """

    def __init__(self, hidden_dim: int, inter_dim: int):
        super().__init__()
        self.w_gate = nn.Linear(hidden_dim, inter_dim, bias=False)
        self.w_up = nn.Linear(hidden_dim, inter_dim, bias=False)
        self.w_down = nn.Linear(inter_dim, hidden_dim, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        # x: [N_tokens, D] → [N_tokens, D]
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class Gate(nn.Module):
    """MoE Router with DeepSeek V3 auxiliary-loss-free load balancing.

    Key design (from DeepSeek V3 official code):
    1. Sigmoid scoring — independent expert probabilities (not competitive softmax)
    2. Group-based routing — top-2-sum per group, select topk_group groups
    3. Bias for selection, original scores for weights — bias steers without corrupting gradients
    4. FP32 for all gating computations — numerical stability
    5. norm_topk_prob — normalize selected weights to sum to 1 (BUG FIX #2)
    6. routed_scaling_factor applied to weights after normalization
    """

    def __init__(
        self,
        hidden_dim: int,
        n_routed_experts: int,
        num_experts_per_tok: int,
        n_group: int,
        topk_group: int,
        scoring_func: str = "sigmoid",
        norm_topk_prob: bool = True,
        routed_scaling_factor: float = 2.5,
        seq_aux_loss_alpha: float = 0.0001,
    ):
        super().__init__()
        self.n_routed_experts = n_routed_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.n_group = n_group
        self.topk_group = topk_group
        self.experts_per_group = n_routed_experts // n_group
        self.scoring_func = scoring_func
        self.norm_topk_prob = norm_topk_prob
        self.routed_scaling_factor = routed_scaling_factor
        self.seq_aux_loss_alpha = seq_aux_loss_alpha

        # Router projection: hidden_dim → n_routed_experts
        self.router_weight = nn.Linear(hidden_dim, n_routed_experts, bias=False)

        # Dynamic bias for auxiliary-loss-free load balancing (NOT a parameter)
        self.register_buffer("bias", torch.zeros(n_routed_experts))

    def forward(self, hidden_states: Tensor) -> Tuple[Tensor, Tensor, Tensor, Dict[str, Tensor]]:
        """Route tokens to experts.

        Args:
            hidden_states: [N, D] (pre-flattened by MoE, N = B*S)

        Returns:
            weights:  [N, K] — routing weights (scaled, normalized original scores)
            indices:  [N, K] — selected expert indices
            aux_loss: scalar — sequence-level auxiliary loss
            metadata: dict with 'load_counts' [E] and 'H_load' scalar
        """
        N = hidden_states.shape[0]
        E = self.n_routed_experts
        K = self.num_experts_per_tok
        G = self.n_group
        EPG = self.experts_per_group
        dtype = hidden_states.dtype
        device = hidden_states.device

        # STEP 1: Raw scores in FP32
        logits = self.router_weight(hidden_states)  # [N, E]
        if self.scoring_func == "sigmoid":
            scores = torch.sigmoid(logits).float()  # [N, E] FP32
        else:
            scores = F.softmax(logits, dim=-1).float()  # [N, E] FP32

        # STEP 2: Group-based routing
        # Score each group by sum of its top-2 expert scores (DeepSeek V3 official pattern)
        scores_grouped = scores.view(N, G, EPG)  # [N, G, EPG]
        group_top2, _ = scores_grouped.topk(2, dim=-1)  # [N, G, 2]
        group_scores = group_top2.sum(dim=-1)  # [N, G]
        _, top_groups = group_scores.topk(self.topk_group, dim=-1)  # [N, topk_group]

        # Build expert mask from selected groups
        group_mask = torch.zeros(N, G, device=device, dtype=scores.dtype)
        group_mask.scatter_(1, top_groups, 1.0)  # [N, G]
        expert_mask = group_mask.unsqueeze(-1).expand(-1, -1, EPG).reshape(N, E)  # [N, E]

        # STEP 3: Biased selection (bias NOT in gradient graph)
        biased = scores + self.bias.unsqueeze(0)  # [N, E]
        biased = biased.masked_fill(expert_mask == 0, float("-inf"))

        # STEP 4: Top-K on biased scores → get indices
        _, indices = biased.topk(K, dim=-1)  # [N, K]

        # STEP 5: Gather ORIGINAL scores for weights (not biased)
        weights = scores.gather(1, indices)  # [N, K]

        # STEP 6: BUG FIX #2 — normalize top-k probabilities
        if self.norm_topk_prob:
            weights = weights / (weights.sum(dim=-1, keepdim=True) + 1e-20)

        # STEP 7: Apply routed_scaling_factor (DeepSeek V3 pattern: scale weights, not output)
        weights = weights * self.routed_scaling_factor

        # STEP 8: Compute auxiliary loss and metadata
        one_hot = torch.zeros(N, E, device=device, dtype=scores.dtype)
        one_hot.scatter_(1, indices, 1.0)
        load_counts = one_hot.sum(dim=0)  # [E]

        # Sequence-level aux loss: L = alpha * mean((f_i - 1/E)^2)
        f = load_counts / max(N, 1)
        target = 1.0 / E
        aux_loss = self.seq_aux_loss_alpha * ((f - target) ** 2).mean()

        # Load balance entropy for monitoring (H_load → higher = more balanced)
        f_safe = f + 1e-10
        H_load = -(f_safe * f_safe.log()).sum()

        metadata = {
            "load_counts": load_counts.detach(),
            "H_load": H_load.detach(),
        }

        # Cast weights back to input dtype
        weights = weights.to(dtype)

        return weights, indices, aux_loss, metadata

    def update_bias(self, load_counts: Tensor, gamma: float) -> None:
        """Update dynamic bias for load balancing (called by training loop).

        Args:
            load_counts: [E] — token counts per expert from forward pass
            gamma: bias update rate (0.0 if frozen)
        """
        if gamma <= 0:
            return
        mean_load = load_counts.float().mean()
        if mean_load > 0:
            self.bias -= gamma * (load_counts.float() - mean_load) / mean_load


class MoE(nn.Module):
    """Mixture of Experts with DeepSeek V3 architecture.

    Components:
    - Gate (router): sigmoid scoring + group routing + bias balancing
    - Routed experts: 64 SwiGLU FFNs, top-8 selected per token
    - Shared expert: 1 SwiGLU MLP with combined inter_dim (n_shared * moe_inter_dim)
      Applied to ALL tokens unconditionally (captures common patterns)

    Token-centric dispatch: iterate over experts, process each expert's batch.
    """

    def __init__(
        self,
        hidden_dim: int,
        moe_inter_dim: int,
        n_routed_experts: int = 64,
        num_experts_per_tok: int = 8,
        n_shared_experts: int = 2,
        n_group: int = 8,
        topk_group: int = 4,
        scoring_func: str = "sigmoid",
        norm_topk_prob: bool = True,
        routed_scaling_factor: float = 2.5,
        seq_aux_loss_alpha: float = 0.0001,
        shared_inter_dim: Optional[int] = None,
    ):
        super().__init__()
        self.n_routed_experts = n_routed_experts
        self.num_experts_per_tok = num_experts_per_tok

        # Gate (router)
        self.gate = Gate(
            hidden_dim=hidden_dim,
            n_routed_experts=n_routed_experts,
            num_experts_per_tok=num_experts_per_tok,
            n_group=n_group,
            topk_group=topk_group,
            scoring_func=scoring_func,
            norm_topk_prob=norm_topk_prob,
            routed_scaling_factor=routed_scaling_factor,
            seq_aux_loss_alpha=seq_aux_loss_alpha,
        )

        # Routed experts (sparse, selected per token)
        self.routed_experts = nn.ModuleList(
            [Expert(hidden_dim, moe_inter_dim) for _ in range(n_routed_experts)]
        )

        # Shared expert (dense, always active) — single MLP with combined inter_dim
        # DeepSeek V3 pattern: n_shared_experts is a multiplier, not a count of modules
        effective_shared_dim = shared_inter_dim or (n_shared_experts * moe_inter_dim)
        self.shared_expert = Expert(hidden_dim, effective_shared_dim)

    def forward(self, hidden_states: Tensor) -> Tuple[Tensor, Dict[str, Tensor]]:
        """Process tokens through routed + shared experts.

        Args:
            hidden_states: [B, S, D]

        Returns:
            output:   [B, S, D]
            aux_data: dict with 'aux_loss', 'load_counts', 'H_load'
        """
        orig_shape = hidden_states.shape  # [B, S, D]
        hidden_dim = orig_shape[-1]
        x = hidden_states.view(-1, hidden_dim)  # [N, D] where N = B*S

        # Route tokens
        weights, indices, aux_loss, metadata = self.gate(x)
        # weights: [N, K], indices: [N, K]

        # Token-centric dispatch for routed experts
        routed_output = torch.zeros_like(x)  # [N, D]
        for expert_idx in range(self.n_routed_experts):
            # Find tokens assigned to this expert across any of the K slots
            mask = (indices == expert_idx)  # [N, K] bool
            token_mask = mask.any(dim=-1)  # [N] bool

            if not token_mask.any():
                continue  # Skip experts with 0 assigned tokens

            # Process this expert's batch
            expert_input = x[token_mask]  # [n_tokens, D]
            expert_out = self.routed_experts[expert_idx](expert_input)  # [n_tokens, D]

            # Sum weights across all slots that selected this expert per token
            # (handles rare case where same expert selected in multiple slots)
            expert_weights = (weights * mask.float()).sum(dim=-1)  # [N]
            active_weights = expert_weights[token_mask]  # [n_tokens]

            routed_output[token_mask] += expert_out * active_weights.unsqueeze(-1)

        # Shared expert — processes ALL tokens
        shared_output = self.shared_expert(x)  # [N, D]

        # Combine routed + shared
        output = routed_output + shared_output  # [N, D]
        output = output.view(*orig_shape)  # [B, S, D]

        aux_data = {
            "aux_loss": aux_loss,
            "load_counts": metadata["load_counts"],
            "H_load": metadata["H_load"],
        }

        return output, aux_data


NameError: name 'Optional' is not defined